# Univariate Logistic Regression From Scratch

## What is Logistic Regression?

Logistic regression is a **binary classification** algorithm.
Given one input feature, it predicts the **probability** that a sample belongs to class 1.

The word 'regression' is misleading — it is a classifier, not a regressor.
The name comes from the fact that it uses a linear equation internally,
but wraps it in a function that squashes the output into a probability.

## What we build step by step:

| Step | What happens |
|------|--------------|
| 1 | Show why linear regression fails for classification |
| 2 | Introduce the sigmoid function — squashes output to (0, 1) |
| 3 | Define the loss function (log-loss) and explain why not MSE |
| 4 | Derive the gradients by hand |
| 5 | Implement gradient descent training loop from scratch |
| 6 | Visualise the sigmoid curve fitted to real data |
| 7 | Compare with sklearn |

## Step 1 — Why Not Linear Regression for Classification?

Suppose we want to predict whether a tumour is benign (1) or malignant (0)
based on its radius.

If we use linear regression, the output can be **any number** — including values
below 0 or above 1. Those are not valid probabilities.

We need a model whose output is always between 0 and 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

# Load data — use only one feature (mean radius) for univariate logistic regression
data = load_breast_cancer()
X_raw = data.data[:, 0]   # mean radius
y     = data.target        # 1 = benign, 0 = malignant

# Show the problem with linear regression
from numpy.polynomial import polynomial as P
coeffs = np.polyfit(X_raw, y, 1)
x_line = np.linspace(X_raw.min() - 2, X_raw.max() + 2, 300)
y_linear = np.polyval(coeffs, x_line)

plt.figure(figsize=(8, 4))
plt.scatter(X_raw, y, alpha=0.3, label='Data points', color='steelblue')
plt.plot(x_line, y_linear, color='tomato', linewidth=2, label='Linear regression')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axhline(1, color='gray', linestyle='--', linewidth=0.8)
plt.fill_between(x_line, 0, 1, alpha=0.05, color='green', label='Valid probability range')
plt.xlabel('Mean Radius')
plt.ylabel('Label (0 or 1)')
plt.title('Problem: Linear regression goes outside [0, 1]')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

out_of_range = np.sum((y_linear < 0) | (y_linear > 1))
print(f'Linear regression predicts values outside [0,1] for {out_of_range} / {len(x_line)} points')

## Step 2 — The Sigmoid Function

We need a function that:
- Takes any real number as input
- Always outputs a value between 0 and 1
- Has an S-shaped curve (smooth transition from 0 to 1)

The **sigmoid function** does exactly this:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

**Intuition:**
- z = 0 → sigmoid = 0.5 (uncertain)
- z very large (positive) → sigmoid ≈ 1 (confident class 1)
- z very large (negative) → sigmoid ≈ 0 (confident class 0)

Our model becomes:
```
z    = w * x + b          (linear combination, same as linear regression)
p    = sigmoid(z)         (squash into probability)
```
We learn `w` (weight) and `b` (bias) from data.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_vals = np.linspace(-10, 10, 300)

plt.figure(figsize=(8, 4))
plt.plot(z_vals, sigmoid(z_vals), linewidth=2.5, color='steelblue')
plt.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0,   color='gray', linestyle='--', linewidth=0.8)
plt.scatter([0], [0.5], color='tomato', zorder=5, s=80, label='z=0 → p=0.5 (uncertain)')
plt.title('The Sigmoid Function: maps any number to (0, 1)')
plt.xlabel('z  (linear output = w*x + b)')
plt.ylabel('sigmoid(z)  =  probability')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print('Sigmoid outputs:')
for z in [-5, -2, 0, 2, 5]:
    print(f'  sigmoid({z:+d}) = {sigmoid(z):.4f}')

## Step 3 — The Loss Function (Log-Loss)

We need a way to measure how wrong our predictions are.

**Why not MSE?**
With the sigmoid inside, the MSE loss surface has many local minima —
gradient descent can get stuck.

**Log-loss (Binary Cross-Entropy)** gives a convex surface — one global minimum,
gradient descent always finds it.

For one sample:
$$
L = -\left[ y \cdot \log(p) + (1 - y) \cdot \log(1 - p) \right]
$$

Where `p = sigmoid(w*x + b)` is our predicted probability.

**Intuition:**
- If y=1 and p=0.99 → loss ≈ 0.01 (nearly right, small penalty)
- If y=1 and p=0.01 → loss ≈ 4.6 (very wrong, huge penalty)
- Confident wrong predictions are punished **much more** than uncertain ones

Over all N samples:
$$
L = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \cdot \log(p_i) + (1 - y_i) \cdot \log(1 - p_i) \right]
$$

In [ ]:
# Visualise log-loss behaviour
p_vals = np.linspace(0.001, 0.999, 300)

loss_y1 = -np.log(p_vals)        # loss when true label = 1
loss_y0 = -np.log(1 - p_vals)    # loss when true label = 0

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(p_vals, loss_y1, color='steelblue', linewidth=2)
axes[0].set_title('Loss when true label = 1\n(want p close to 1)')
axes[0].set_xlabel('Predicted probability p')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(p_vals, loss_y0, color='tomato', linewidth=2)
axes[1].set_title('Loss when true label = 0\n(want p close to 0)')
axes[1].set_xlabel('Predicted probability p')
axes[1].set_ylabel('Loss')
axes[1].grid(True)

plt.suptitle('Log-Loss: confident wrong predictions are punished most', fontsize=12)
plt.tight_layout()
plt.show()

## Step 4 — Deriving the Gradients

To train with gradient descent we need to know:
how does the loss change when we change `w` or `b`?

After applying the chain rule (the math works out surprisingly cleanly):

$$
\frac{\partial L}{\partial w} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i) \cdot x_i
$$

$$
\frac{\partial L}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)
$$

**Why this is elegant:**
The gradient is just the average error `(p - y)` — how far off we are.
For `w`, we weight each error by its input `x`.

**Update rule (gradient descent):**
```
w = w - lr * dL/dw
b = b - lr * dL/db
```

## Step 5 — Training From Scratch

We implement the full logistic regression training loop:
forward pass → compute loss → compute gradients → update weights → repeat.

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, epochs=1000):
        self.lr     = lr
        self.epochs = epochs
        self.w      = 0.0
        self.b      = 0.0
        self.losses = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # clip prevents overflow

    def fit(self, X, y):
        N = len(X)
        for epoch in range(self.epochs):
            # Forward pass
            z = self.w * X + self.b
            p = self._sigmoid(z)

            # Log-loss
            loss = -np.mean(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15))
            self.losses.append(loss)

            # Gradients
            error = p - y
            dw = np.mean(error * X)
            db = np.mean(error)

            # Update
            self.w -= self.lr * dw
            self.b -= self.lr * db

            if (epoch + 1) % 200 == 0:
                print(f'Epoch {epoch+1:5d}  |  Loss: {loss:.4f}  |  w: {self.w:.4f}  b: {self.b:.4f}')

    def predict_proba(self, X):
        return self._sigmoid(self.w * X + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# Normalise X so gradient descent converges smoothly
X_mean = X_raw.mean()
X_std  = X_raw.std()
X_norm = (X_raw - X_mean) / X_std

model_scratch = LogisticRegressionScratch(lr=0.1, epochs=1000)
model_scratch.fit(X_norm, y)

## Step 6 — Loss Curve

The loss should decrease steadily as gradient descent finds better weights.
If the loss is not decreasing, the learning rate is too high or too low.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model_scratch.losses)
plt.title('Log-Loss Over Training Epochs')
plt.xlabel('Epoch')
plt.ylabel('Log-Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Initial loss: {model_scratch.losses[0]:.4f}')
print(f'Final loss:   {model_scratch.losses[-1]:.4f}')
print(f'Learned w = {model_scratch.w:.4f},  b = {model_scratch.b:.4f}')

## Step 7 — Visualise the Fitted Sigmoid Curve

The fitted sigmoid curve shows the predicted probability of being benign
as a function of mean radius.

The decision boundary is where the curve crosses 0.5 —
samples to the right of this point are predicted benign,
samples to the left are predicted malignant.

In [ ]:
x_plot_norm = np.linspace(X_norm.min(), X_norm.max(), 300)
p_plot      = model_scratch.predict_proba(x_plot_norm)

# Convert back to original scale for the x-axis label
x_plot_orig = x_plot_norm * X_std + X_mean

# Decision boundary in original scale
boundary_norm = -model_scratch.b / model_scratch.w
boundary_orig = boundary_norm * X_std + X_mean

plt.figure(figsize=(9, 5))
plt.scatter(X_raw[y==0], y[y==0], alpha=0.3, color='tomato',    label='Malignant (0)', s=20)
plt.scatter(X_raw[y==1], y[y==1], alpha=0.3, color='steelblue', label='Benign (1)',    s=20)
plt.plot(x_plot_orig, p_plot, color='black', linewidth=2.5, label='Sigmoid curve (our model)')
plt.axvline(boundary_orig, color='green', linestyle='--', linewidth=1.5,
            label=f'Decision boundary (radius = {boundary_orig:.1f})')
plt.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
plt.xlabel('Mean Radius')
plt.ylabel('P(Benign)')
plt.title('Logistic Regression: Fitted Sigmoid Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Decision boundary: mean radius = {boundary_orig:.2f}')
print(f'Tumours with radius > {boundary_orig:.1f} predicted BENIGN')
print(f'Tumours with radius < {boundary_orig:.1f} predicted MALIGNANT')

## Step 8 — Accuracy and Threshold

We classify as 1 (benign) if predicted probability >= 0.5, else 0 (malignant).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_scratch = model_scratch.predict(X_norm)

print('Our model results:')
print(f'  Accuracy: {accuracy_score(y, y_pred_scratch):.4f}')
print()
print(classification_report(y, y_pred_scratch, target_names=['Malignant', 'Benign']))

## Step 9 — Compare With sklearn

sklearn's `LogisticRegression` uses the same math but with more optimisations.
Our learned weights should produce similar accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression

sk_model = LogisticRegression(max_iter=1000)
sk_model.fit(X_norm.reshape(-1, 1), y)

y_pred_sk = sk_model.predict(X_norm.reshape(-1, 1))

print(f'Our model   accuracy: {accuracy_score(y, y_pred_scratch):.4f}')
print(f'sklearn     accuracy: {accuracy_score(y, y_pred_sk):.4f}')
print()
print(f'Our weights:    w = {model_scratch.w:.4f},  b = {model_scratch.b:.4f}')
print(f'sklearn weights: w = {sk_model.coef_[0][0]:.4f},  b = {sk_model.intercept_[0]:.4f}')

## Summary

| Concept | Formula | Why |
|---------|---------|-----|
| Linear step | z = w*x + b | Same as linear regression |
| Sigmoid | p = 1 / (1 + e^-z) | Squashes z into a probability (0, 1) |
| Log-loss | -mean(y*log(p) + (1-y)*log(1-p)) | Convex — gradient descent works reliably |
| Gradient for w | mean((p - y) * x) | How much each weight contributed to the error |
| Gradient for b | mean(p - y) | Average error across all samples |
| Update | w = w - lr * dw | Move weights to reduce loss |

## Key Insights

1. **Logistic regression is linear regression + sigmoid.** The decision boundary is still a straight line.
2. **The gradient is just the average error.** `(p - y)` — beautifully simple, comes from the log-loss.
3. **Feature normalisation matters.** Without it, gradient descent is slow or unstable.
4. **The 0.5 threshold is a choice.** In medical diagnosis you might lower it to 0.3 to catch more malignant cases (higher recall).